# sum-and-broadcast-duality composite — cx16: sum-normalize each row to a probability distribution

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `sum-and-broadcast-duality`, `broadcasting-rules`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "sum-and-broadcast-duality"
DD_ATOM_IDS = ["sum-and-broadcast-duality", "broadcasting-rules"]
DD_SUBTOPICS = ["Backprop: sum/broadcast duality", "Numpy: Vectorization and broadcasting"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sum-normalize = divide each row by its row sum

1. **`sum-and-broadcast-duality`** — `x.sum(dim=-1, keepdim=True)` collapses
   the trailing axis to size 1. The kept axis is precisely the shape the
   divide step needs to broadcast back.
2. **`broadcasting-rules`** — the `(B, 1)` divisor right-aligns
   against `(B, D)` and the size-1 axis is stretched (zero-stride)
   across the D axis. The keepdim=True step exists to set up this
   broadcast — no explicit `.expand()` needed.

Composition: this is what turns a vector of non-negative counts into a
probability distribution (each row sums to 1). It also generalises
`F.normalize(x, p=1)` — the L1 row-normalize.


### Composite Exercise — sum-normalize each row to a probability distribution

**Atoms exercised together**: `sum-and-broadcast-duality`, `broadcasting-rules`

Implement `cx16_sum_normalize(x)`. Given `x` shape `(B, D)` of non-negative
values (the test feeds only non-negative inputs):

1. Compute per-row sum with `keepdim=True` → shape `(B, 1)`.
2. Divide `x` by that sum; broadcasting expands `(B, 1)` over `D`.
3. Return shape `(B, D)` where every row sums to `1.0` (within float tol).

Constraints:
- Use `sum(..., keepdim=True)` — the whole point is the sum+keepdim pattern.
- No `F.normalize`, no manual division by `x.sum(-1).unsqueeze(-1)`. The
  keepdim flag IS the drill.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx16_sum_normalize(x: Tensor) -> Tensor:
    """Divide each row by its row sum so each row sums to 1."""
    raise NotImplementedError()


def _test_cx16():
    import inspect
    src = inspect.getsource(cx16_sum_normalize)
    assert 'keepdim' in src, 'must use keepdim=True — that\'s the atom'
    assert 'F.normalize' not in src, 'do not use F.normalize'

    # --- canonical small batch: rows sum to 1 ---
    x = t.tensor([[1.0, 1.0, 2.0],
                  [3.0, 1.0, 6.0],
                  [0.5, 0.5, 0.0]])
    out = cx16_sum_normalize(x)
    assert out.shape == x.shape
    expected = t.tensor([[0.25, 0.25, 0.5],
                         [0.3, 0.1, 0.6],
                         [0.5, 0.5, 0.0]])
    assert t.allclose(out, expected, atol=1e-6), f'got {out}'

    # --- random non-negative batch: each row sums to 1 ---
    rng = t.Generator().manual_seed(0)
    X = t.rand(7, 11, generator=rng) + 0.01
    Y = cx16_sum_normalize(X)
    row_sums = Y.sum(dim=-1)
    assert t.allclose(row_sums, t.ones(7), atol=1e-5), f'row sums: {row_sums}'

    # --- value witness: matches x / x.sum(dim=-1, keepdim=True) ---
    ref = X / X.sum(dim=-1, keepdim=True)
    assert t.allclose(Y, ref, atol=1e-6)

    # --- shape preserved (non-square D) ---
    Y2 = cx16_sum_normalize(t.rand(4, 9, generator=rng) + 0.01)
    assert Y2.shape == (4, 9)
    assert t.allclose(Y2.sum(dim=-1), t.ones(4), atol=1e-5)

    _dd_passed.add('cx16')

_test_cx16()

<details><summary>Show solution — cx16</summary>

```python
def cx16_sum_normalize(x: Tensor) -> Tensor:
    # atoms compose: sum(keepdim=True) is the sum-and-broadcast-duality
    # step; the divide is the broadcasting-rules step — (B, 1) right-aligns
    # against (B, D) and the size-1 axis is stretched (zero-stride). The (B, 1) divisor right-aligns against D and expands.
    return x / x.sum(dim=-1, keepdim=True)

```

Pattern is identical to L2 row-normalize — just swap `.norm` for `.sum`.
The keepdim=True is doing all the broadcasting work. This is also exactly
what `F.normalize(x, p=1, dim=-1)` produces.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["Backprop: sum/broadcast duality", "Numpy: Vectorization and broadcasting"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()